# Week 5, Lab 5 — Mini-project (your framework)

Same brief as Week 4 Lab 5, implemented in AutoGen or Pydantic AI.


In [ ]:
WEEK = 'Week 5'
LAB = 'Lab 5 — mini-project'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn pyautogen pydantic-ai openai
else:
    %pip install -q pyautogen pydantic-ai ollama openai


In [ ]:
from pydantic import BaseModel
from pydantic_ai import Agent

cfg = openai_client_kwargs()
try:
    from pydantic_ai.models.openai import OpenAIChatModel
    from pydantic_ai.providers.openai import OpenAIProvider
    model = OpenAIChatModel(cfg["model"], provider=OpenAIProvider(base_url=cfg["base_url"], api_key=cfg["api_key"]))
except Exception:
    from pydantic_ai.models.openai import OpenAIModel
    model = OpenAIModel(cfg["model"], base_url=cfg["base_url"], api_key=cfg["api_key"])

class Plan(BaseModel):
    route: str
    expression: str | None = None
    topic: str | None = None

planner = Agent(model, output_type=Plan, instructions="route is math or research")
answerer = Agent(model, instructions="Write a short final answer using the tool result.")

def solve(question: str) -> str:
    plan = planner.run_sync(question).output
    if plan.route == "math":
        tool = calculator(plan.expression or "0")
    else:
        tool = lookup_fact(plan.topic or question)
    print("plan:", plan, "tool:", tool)
    ok = input("Approve? [y/n] ").strip().lower() != "n"
    if not ok:
        return "Human rejected the tool result."
    return str(answerer.run_sync(f"Q: {question}\nTool: {tool}").output)

print(solve("What is MCP?"))


Add a short reflection cell: which framework you would use at work and why.
